In [ ]:
import kagglehub
gold_price_prediction_dataset_path = kagglehub.dataset_download('sid321axn/gold-price-prediction-dataset')
print('Data source import complete.')

In [ ]:
import pandas as pd
df = pd.read_csv(gold_price_prediction_dataset_path+'/FINAL_USO.csv')
df.head()

# Date column

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')
df.head()

# Stationarity check



In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_test = adfuller(df['Close'])
print('p-value: %f' % adf_test[1])
if adf_test[1] < 0.05:
    print('The time series is likely stationary.')
else:
    print('The time series is likely non-stationary.')

# Implement AR, MA, ARMA, and ARIMA models


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import train_test_split

close_price = df['Close']
train_data, test_data = train_test_split(close_price, test_size=0.2, shuffle=False)

In [ ]:
# Fit AR model
ar_model = ARIMA(train_data, order=(1, 0, 0))
ar_results = ar_model.fit()
ar_predictions = ar_results.predict(start=len(train_data), end=len(df)-1)

# Fit MA model
ma_model = ARIMA(train_data, order=(0, 0, 1))
ma_results = ma_model.fit()
ma_predictions = ma_results.predict(start=len(train_data), end=len(df)-1)

# Fit ARMA model
arma_model = ARIMA(train_data, order=(1, 0, 1))
arma_results = arma_model.fit()
arma_predictions = arma_results.predict(start=len(train_data), end=len(df)-1)

# Fit ARIMA model
# Since the ADF test indicated non-stationarity (p-value > 0.05),
# we will use differencing (d=1) for the ARIMA model.
# We can analyze ACF and PACF of the differenced series to determine p and q,
# but for simplicity, we'll start with (1, 1, 1) as a common starting point.
arima_model = ARIMA(train_data, order=(1, 1, 1))
arima_results = arima_model.fit()
arima_predictions = arima_results.predict(start=len(train_data), end=len(df)-1)

print("AR Predictions:")
print(ar_predictions.head())
print("\nMA Predictions:")
print(ma_predictions.head())
print("\nARMA Predictions:")
print(arma_predictions.head())
print("\nARIMA Predictions:")
print(arima_predictions.head())

## Implement SARIMA



In [ ]:

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(train_data, order=(1, 1, 1), seasonal_order=(1, 0, 0, 12))
sarima_results = sarima_model.fit()

sarima_predictions = sarima_results.predict(start=len(train_data), end=len(df)-1)
print("SARIMA Predictions:")
print(sarima_predictions.head())

# Implement exponential smoothing


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

exp_smoothing_model = ExponentialSmoothing(train_data, trend='add', seasonal='add', seasonal_periods=12)
exp_smoothing_results = exp_smoothing_model.fit()


exp_smoothing_predictions = exp_smoothing_results.predict(start=len(train_data), end=len(df)-1)
print("Exponential Smoothing Predictions:")
print(exp_smoothing_predictions.head())

# Implement linear trend model


In [ ]:
import numpy as np

In [ ]:
from sklearn.linear_model import LinearRegression

train_time_steps = np.arange(len(train_data)).reshape(-1, 1)
linear_trend_model = LinearRegression()

linear_trend_model.fit(train_time_steps, train_data)
test_time_steps = np.arange(len(train_data), len(df)).reshape(-1, 1)

linear_trend_predictions = linear_trend_model.predict(test_time_steps)
print("Linear Trend Predictions:")
print(linear_trend_predictions[:5])

# Implement prophet


In [ ]:
from prophet import Prophet
from sklearn.model_selection import ParameterGrid
from prophet.diagnostics import cross_validation, performance_metrics

prophet_train_df = train_data.reset_index().rename(columns={'Date': 'ds', 'Close': 'y'})
param_grid = {
    'changepoint_prior_scale': [0.001, 0.01, 0.1, 0.5],
    'seasonality_prior_scale': [0.01, 0.1, 1.0, 10.0],
    'n_changepoints': [5, 10, 20, 25]
}

all_params = list(ParameterGrid(param_grid))
rmses = []

for params in all_params:
    prophet_model = Prophet(**params)
    prophet_model.fit(prophet_train_df)
    df_cv = cross_validation(prophet_model, initial='730 days', period='180 days', horizon='365 days')
    df_p = performance_metrics(df_cv, metrics=['rmse'])
    rmses.append(df_p['rmse'].mean())

best_params = all_params[np.argmin(rmses)]
print(f"Best parameters: {best_params}")

prophet_model = Prophet(**best_params)
prophet_model.fit(prophet_train_df)

future = prophet_model.make_future_dataframe(periods=len(test_data))

prophet_predictions = prophet_model.predict(future)['yhat']

print("\nProphet Predictions (with best parameters):")
print(prophet_predictions.head())

# Evaluate models


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt

def evaluate_model(actual, predicted):
    rmse = sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return rmse, mae

evaluation_results = {}

In [ ]:
start_index = len(train_data)
end_index = len(df) - 1

ar_predictions = ar_results.predict(start=start_index, end=end_index)
ma_predictions = ma_results.predict(start=start_index, end=end_index)
arma_predictions = arma_results.predict(start=start_index, end=end_index)
arima_predictions = arima_results.predict(start=start_index, end=end_index)
sarima_predictions = sarima_results.predict(start=start_index, end=end_index)
exp_smoothing_predictions = exp_smoothing_results.predict(start=start_index, end=end_index)

ar_rmse, ar_mae = evaluate_model(test_data, ar_predictions)
evaluation_results['AR'] = {'RMSE': ar_rmse, 'MAE': ar_mae}

ma_rmse, ma_mae = evaluate_model(test_data, ma_predictions)
evaluation_results['MA'] = {'RMSE': ma_rmse, 'MAE': ma_mae}

arma_rmse, arma_mae = evaluate_model(test_data, arma_predictions)
evaluation_results['ARMA'] = {'RMSE': arma_rmse, 'MAE': arma_mae}

arima_rmse, arima_mae = evaluate_model(test_data, arima_predictions)
evaluation_results['ARIMA'] = {'RMSE': arima_rmse, 'MAE': arima_mae}

sarima_rmse, sarima_mae = evaluate_model(test_data, sarima_predictions)
evaluation_results['SARIMA'] = {'RMSE': sarima_rmse, 'MAE': sarima_mae}

exp_smoothing_rmse, exp_smoothing_mae = evaluate_model(test_data, exp_smoothing_predictions)
evaluation_results['Exponential Smoothing'] = {'RMSE': exp_smoothing_rmse, 'MAE': exp_smoothing_mae}

linear_trend_rmse, linear_trend_mae = evaluate_model(test_data, linear_trend_predictions)
evaluation_results['Linear Trend'] = {'RMSE': linear_trend_rmse, 'MAE': linear_trend_mae}

# Select only the predictions corresponding to the test data for Prophet
prophet_test_predictions = prophet_predictions.iloc[-len(test_data):]
prophet_rmse, prophet_mae = evaluate_model(test_data, prophet_test_predictions)
evaluation_results['Prophet'] = {'RMSE': prophet_rmse, 'MAE': prophet_mae}

# Compare results


In [ ]:
evaluation_df = pd.DataFrame.from_dict(evaluation_results, orient='index')
evaluation_df = evaluation_df.sort_values(by=['RMSE', 'MAE'])

print("Model Performance Summary (Sorted by RMSE):")
display(evaluation_df)  # Without tuning Prophet MAE was 11 and now 8.2

# Visualize predictions

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))
plt.plot(test_data.index, test_data, label='Actual Prices')
plt.plot(test_data.index, ar_predictions, label='AR Predictions')
plt.plot(test_data.index, ma_predictions, label='MA Predictions')
plt.plot(test_data.index, arma_predictions, label='ARMA Predictions')
plt.plot(test_data.index, arima_predictions, label='ARIMA Predictions')
plt.plot(test_data.index, sarima_predictions, label='SARIMA Predictions')
plt.plot(test_data.index, exp_smoothing_predictions, label='Exponential Smoothing Predictions')
plt.plot(test_data.index, linear_trend_predictions, label='Linear Trend Predictions')
plt.plot(test_data.index, prophet_test_predictions, label='Prophet Predictions')


plt.title('Gold Price Prediction - Actual vs. Predicted')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.grid(True)
plt.show()